# Current Coverage Residual Guard Refresh v1

Direction: `execution` diagnostic.  
Role: `diagnose`; no runtime promotion from this notebook.  
Changed layer: path-level residual drawdown guard applied to current full-coverage daily paths.  
Primary question: can the old low-drawdown residual guard mechanism be refreshed to `2026-05-21` coverage before using it as a base for state-conditioned reward/router experiments? The screen is intentionally compressed around the old successful guard and the current lowest-drawdown full-coverage families to reduce backtest-overfitting risk.

## Governance

- Canonical baseline: current scheduled/paper family around `balanced_rule_residual_v22` and `paper_model_oos_risk_check_v1`.
- Direct comparison: old `risk_budget_residual_drawdown_guard_v1`, which cleared the strict target but only through `2026-03-31`.
- Role: diagnose stale evidence and rebuild the strict/WF low-DD base if possible.
- Target: strict full coverage `relative_return >= +30pp` and `max_dd <= 15%`, plus walk-forward guards.
- Comparison-table row key: `current_coverage_residual_guard_refresh_v1`.

## Historical reflection

- Closest success: `risk_budget_residual_drawdown_guard_v1` selected `+30.46pp` / `14.64% MaxDD`, but only through `2026-03-31`.
- Closest failure: `state_conditioned_rebound_risk_router_v1` showed that using the stale low-DD base on common dates produced no strict/full pass and explicitly warned that the residual base must be refreshed.
- Why this is not a duplicate: this notebook starts from current full-coverage paths ending `2026-05-21`, screens the residual guard across multiple current families, and carries the same config into saved walk-forward windows.

## External basis

- Qlib keeps the loop in a reproducible AI quant harness and research-to-production workflow: https://github.com/microsoft/qlib
- Chekhlov, Uryasev, and Zabarankin motivate drawdown as a path-dependent underwater-curve risk measure: https://papers.ssrn.com/sol3/Delivery.cfm/SSRN_ID544742_code375170.pdf?abstractid=544742&mirid=1
- Bailey et al. motivate limiting configuration search and requiring out-of-sample/walk-forward checks against backtest overfitting: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2308659
- Deep Momentum Networks motivate separating trend estimation/position sizing and accounting for turnover: https://arxiv.org/abs/1904.04912

## Acceptance Criteria

Full pass requires:

- strict full coverage: `relative_return >= 30pp` and `max_dd <= 15%`;
- walk-forward windows: `max_dd <= 15%` for each of `wf_2020`, `wf_2022`, `wf_2023_2024`;
- walk-forward relative-return floors: `wf_2020 >= -10pp`, `wf_2022 >= 30pp`, `wf_2023_2024 >= 30pp`.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
from itertools import product
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "runtime").exists():
            return candidate
    raise RuntimeError(f"could not locate repo root from {start}")


ROOT = find_repo_root()
EXPERIMENT = "current_coverage_residual_guard_refresh_v1"
CARD = ROOT / "runtime/execution/card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

STARTING_NAV = 1_000_000.0
STRICT_TARGET_REL = 30.0
STRICT_TARGET_DD = 15.0
WF_REL_FLOORS = {"wf_2020": -10.0, "wf_2022": 30.0, "wf_2023_2024": 30.0}
WINDOWS = ["strict_oos", "wf_2020", "wf_2022", "wf_2023_2024"]

FAMILIES = {
    "paper_model_oos_risk_check_v1": {
        "strict_oos": ROOT / "runtime/production/card/paper_model_oos_risk_check_v1/paper_model_oos_risk_check_v1_soxl_daily.csv",
    },
    "recovery_reentry_2021_trendbreak_cap_v1": {
        "strict_oos": ROOT / "runtime/execution/card/recovery_reentry_2021_trendbreak_cap_v1/recovery_reentry_2021_trendbreak_cap_v1_soxl_daily.csv",
        "wf_2020": ROOT / "runtime/execution/card/recovery_reentry_2021_trendbreak_cap_v1/walkforward/wf_2020/recovery_reentry_2021_trendbreak_cap_v1_wf_2020_soxl_daily.csv",
        "wf_2022": ROOT / "runtime/execution/card/recovery_reentry_2021_trendbreak_cap_v1/walkforward/wf_2022/recovery_reentry_2021_trendbreak_cap_v1_wf_2022_soxl_daily.csv",
        "wf_2023_2024": ROOT / "runtime/execution/card/recovery_reentry_2021_trendbreak_cap_v1/walkforward/wf_2023_2024/recovery_reentry_2021_trendbreak_cap_v1_wf_2023_2024_soxl_daily.csv",
    },
    "recovery_reentry_anchor_confirm_v1": {
        "strict_oos": ROOT / "runtime/execution/card/recovery_reentry_anchor_confirm_v1/recovery_reentry_anchor_confirm_v1_soxl_daily.csv",
        "wf_2020": ROOT / "runtime/execution/card/recovery_reentry_anchor_confirm_v1/walkforward/wf_2020/recovery_reentry_anchor_confirm_v1_wf_2020_soxl_daily.csv",
        "wf_2022": ROOT / "runtime/execution/card/recovery_reentry_anchor_confirm_v1/walkforward/wf_2022/recovery_reentry_anchor_confirm_v1_wf_2022_soxl_daily.csv",
        "wf_2023_2024": ROOT / "runtime/execution/card/recovery_reentry_anchor_confirm_v1/walkforward/wf_2023_2024/recovery_reentry_anchor_confirm_v1_wf_2023_2024_soxl_daily.csv",
    },
    "recovery_reentry_conditional_vol_target_v1": {
        "strict_oos": ROOT / "runtime/execution/card/recovery_reentry_conditional_vol_target_v1/recovery_reentry_conditional_vol_target_v1_soxl_daily.csv",
        "wf_2020": ROOT / "runtime/execution/card/recovery_reentry_conditional_vol_target_v1/walkforward/wf_2020/recovery_reentry_conditional_vol_target_v1_wf_2020_soxl_daily.csv",
        "wf_2022": ROOT / "runtime/execution/card/recovery_reentry_conditional_vol_target_v1/walkforward/wf_2022/recovery_reentry_conditional_vol_target_v1_wf_2022_soxl_daily.csv",
        "wf_2023_2024": ROOT / "runtime/execution/card/recovery_reentry_conditional_vol_target_v1/walkforward/wf_2023_2024/recovery_reentry_conditional_vol_target_v1_wf_2023_2024_soxl_daily.csv",
    },
    "recovery_reentry_cppi_floor_v1": {
        "strict_oos": ROOT / "runtime/execution/card/recovery_reentry_cppi_floor_v1/recovery_reentry_cppi_floor_v1_soxl_daily.csv",
        "wf_2020": ROOT / "runtime/execution/card/recovery_reentry_cppi_floor_v1/walkforward/wf_2020/recovery_reentry_cppi_floor_v1_wf_2020_soxl_daily.csv",
        "wf_2022": ROOT / "runtime/execution/card/recovery_reentry_cppi_floor_v1/walkforward/wf_2022/recovery_reentry_cppi_floor_v1_wf_2022_soxl_daily.csv",
        "wf_2023_2024": ROOT / "runtime/execution/card/recovery_reentry_cppi_floor_v1/walkforward/wf_2023_2024/recovery_reentry_cppi_floor_v1_wf_2023_2024_soxl_daily.csv",
    },
    "recovery_reentry_equity_drawdown_guard_v1": {
        "strict_oos": ROOT / "runtime/execution/card/recovery_reentry_equity_drawdown_guard_v1/recovery_reentry_equity_drawdown_guard_v1_soxl_daily.csv",
        "wf_2020": ROOT / "runtime/execution/card/recovery_reentry_equity_drawdown_guard_v1/walkforward/wf_2020/recovery_reentry_equity_drawdown_guard_v1_wf_2020_soxl_daily.csv",
        "wf_2022": ROOT / "runtime/execution/card/recovery_reentry_equity_drawdown_guard_v1/walkforward/wf_2022/recovery_reentry_equity_drawdown_guard_v1_wf_2022_soxl_daily.csv",
        "wf_2023_2024": ROOT / "runtime/execution/card/recovery_reentry_equity_drawdown_guard_v1/walkforward/wf_2023_2024/recovery_reentry_equity_drawdown_guard_v1_wf_2023_2024_soxl_daily.csv",
    },
    "recovery_reentry_floor_v1": {
        "strict_oos": ROOT / "runtime/execution/card/recovery_reentry_floor_v1/recovery_reentry_floor_v1_soxl_daily.csv",
        "wf_2020": ROOT / "runtime/execution/card/recovery_reentry_floor_v1/walkforward/wf_2020/recovery_reentry_floor_v1_wf_2020_soxl_daily.csv",
        "wf_2022": ROOT / "runtime/execution/card/recovery_reentry_floor_v1/walkforward/wf_2022/recovery_reentry_floor_v1_wf_2022_soxl_daily.csv",
        "wf_2023_2024": ROOT / "runtime/execution/card/recovery_reentry_floor_v1/walkforward/wf_2023_2024/recovery_reentry_floor_v1_wf_2023_2024_soxl_daily.csv",
    },
    "recovery_reentry_regime_drawdown_guard_v1": {
        "strict_oos": ROOT / "runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/recovery_reentry_regime_drawdown_guard_v1_soxl_daily.csv",
        "wf_2020": ROOT / "runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/walkforward/wf_2020/recovery_reentry_regime_drawdown_guard_v1_wf_2020_soxl_daily.csv",
        "wf_2022": ROOT / "runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/walkforward/wf_2022/recovery_reentry_regime_drawdown_guard_v1_wf_2022_soxl_daily.csv",
        "wf_2023_2024": ROOT / "runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/walkforward/wf_2023_2024/recovery_reentry_regime_drawdown_guard_v1_wf_2023_2024_soxl_daily.csv",
    },
    "soxl_soxx_efficiency_chop_guard_v1": {
        "strict_oos": ROOT / "runtime/execution/card/soxl_soxx_efficiency_chop_guard_v1/soxl_soxx_efficiency_chop_guard_v1_soxl_daily.csv",
        "wf_2020": ROOT / "runtime/execution/card/soxl_soxx_efficiency_chop_guard_v1/walkforward/wf_2020/soxl_soxx_efficiency_chop_guard_v1_wf_2020_soxl_daily.csv",
        "wf_2022": ROOT / "runtime/execution/card/soxl_soxx_efficiency_chop_guard_v1/walkforward/wf_2022/soxl_soxx_efficiency_chop_guard_v1_wf_2022_soxl_daily.csv",
        "wf_2023_2024": ROOT / "runtime/execution/card/soxl_soxx_efficiency_chop_guard_v1/walkforward/wf_2023_2024/soxl_soxx_efficiency_chop_guard_v1_wf_2023_2024_soxl_daily.csv",
    },
}

missing = []
for family, paths in FAMILIES.items():
    missing.extend(str(p.relative_to(ROOT)) for p in paths.values() if not p.exists())
if missing:
    raise FileNotFoundError("missing required paths:" + chr(10) + chr(10).join(missing))

print(f"repo={ROOT}")
print(f"families={len(FAMILIES)}")


repo=/Users/rziiiii/.openclaw/workspace
families=9


In [2]:
def load_daily(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df.sort_values("datetime").reset_index(drop=True)
    required = {"datetime", "price", "position"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path} missing {missing}")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["position"] = pd.to_numeric(df["position"], errors="coerce").clip(0.0, 1.0)
    return df.dropna(subset=["price", "position"]).reset_index(drop=True)


def nav_from_positions(frame: pd.DataFrame, positions: np.ndarray, label: str) -> pd.DataFrame:
    out = frame.copy()
    prices = out["price"].astype(float).to_numpy()
    positions = np.asarray(positions, dtype=float)
    nav = np.empty(len(out), dtype=float)
    nav[0] = STARTING_NAV
    for i in range(1, len(out)):
        nav[i] = nav[i - 1] * (1.0 + positions[i - 1] * (prices[i] / prices[i - 1] - 1.0))
    out["position"] = positions
    out["strategy_nav"] = nav
    out["buyhold_nav"] = STARTING_NAV * prices / prices[0]
    delta = pd.Series(positions).diff().fillna(0.0)
    out["action"] = np.where(delta > 1e-9, "BUY", np.where(delta < -1e-9, "SELL", "HOLD"))
    out["source_label"] = label
    return out


def max_drawdown_pct(nav: pd.Series) -> float:
    nav = nav.astype(float)
    return float((1.0 - nav / nav.cummax()).max() * 100.0)


def max_drawdown_span(frame: pd.DataFrame, nav_col: str = "strategy_nav") -> dict:
    nav = frame[nav_col].astype(float).reset_index(drop=True)
    dd = nav / nav.cummax() - 1.0
    trough_idx = int(dd.idxmin())
    peak_idx = int(nav.loc[:trough_idx].idxmax())
    return {
        "peak_date": frame.loc[peak_idx, "datetime"].date().isoformat(),
        "trough_date": frame.loc[trough_idx, "datetime"].date().isoformat(),
        "max_dd": float(abs(dd.iloc[trough_idx]) * 100.0),
        "peak_position": float(frame.loc[peak_idx, "position"]),
        "trough_position": float(frame.loc[trough_idx, "position"]),
    }


def metrics(frame: pd.DataFrame, label: str) -> dict:
    nav = frame["strategy_nav"].astype(float)
    bh = frame["buyhold_nav"].astype(float)
    prices = frame["price"].astype(float).to_numpy()
    positions = frame["position"].astype(float).to_numpy()
    returns = prices[1:] / prices[:-1] - 1.0
    strat_returns = positions[:-1] * returns if len(positions) > 1 else np.array([])
    sharpe = float(np.mean(strat_returns) / np.std(strat_returns) * np.sqrt(252)) if len(strat_returns) > 2 and np.std(strat_returns) > 0 else 0.0
    changes = np.abs(np.diff(positions))
    return {
        "label": label,
        "rows": int(len(frame)),
        "start": frame["datetime"].iloc[0].date().isoformat(),
        "end": frame["datetime"].iloc[-1].date().isoformat(),
        "return": float((nav.iloc[-1] / nav.iloc[0] - 1.0) * 100.0),
        "buy_and_hold_return": float((bh.iloc[-1] / bh.iloc[0] - 1.0) * 100.0),
        "relative_return": float((nav.iloc[-1] / nav.iloc[0] - bh.iloc[-1] / bh.iloc[0]) * 100.0),
        "sharpe": sharpe,
        "max_dd": max_drawdown_pct(nav),
        "avg_pos": float(np.mean(positions)),
        "trades": int(np.sum(changes > 0.001)),
    }


def score_candidate(strict: dict, wf: dict[str, dict]) -> tuple[float, bool, bool]:
    strict_pass = strict["relative_return"] >= STRICT_TARGET_REL and strict["max_dd"] <= STRICT_TARGET_DD
    strict_loss = max(0.0, STRICT_TARGET_REL - strict["relative_return"]) ** 2 + 4.0 * max(0.0, strict["max_dd"] - STRICT_TARGET_DD) ** 2
    wf_pass = True
    wf_loss = 0.0
    for name, floor in WF_REL_FLOORS.items():
        m = wf.get(name)
        if m is None:
            wf_pass = False
            wf_loss += 500.0
            continue
        wf_pass = wf_pass and m["max_dd"] <= STRICT_TARGET_DD and m["relative_return"] >= floor
        wf_loss += 2.0 * max(0.0, m["max_dd"] - STRICT_TARGET_DD) ** 2
        wf_loss += 0.25 * max(0.0, floor - m["relative_return"]) ** 2
    return float(strict_loss + wf_loss), bool(strict_pass), bool(strict_pass and wf_pass)


@dataclass(frozen=True)
class ResidualGuardConfig:
    start_drawdown: float
    full_drawdown: float
    min_cap: float
    trade_band: float
    release_drawdown: float


def apply_residual_guard(frame: pd.DataFrame, config: ResidualGuardConfig) -> tuple[pd.DataFrame, dict]:
    source = frame.reset_index(drop=True).copy()
    base_pos = source["position"].astype(float).to_numpy()
    prices = source["price"].astype(float).to_numpy()
    positions = np.empty(len(source), dtype=float)
    account_dd = np.zeros(len(source), dtype=float)
    guard_cap = np.ones(len(source), dtype=float)
    guard_active = np.zeros(len(source), dtype=bool)
    nav = 1.0
    peak = 1.0
    current = float(base_pos[0])
    active_days = 0
    changed_days = 0
    for i in range(len(source)):
        if i > 0:
            nav *= 1.0 + current * (prices[i] / prices[i - 1] - 1.0)
            peak = max(peak, nav)
        dd = 1.0 - nav / peak if peak else 0.0
        account_dd[i] = dd
        if dd <= config.release_drawdown:
            desired = base_pos[i]
            cap = 1.0
        elif dd >= config.start_drawdown:
            denom = max(config.full_drawdown - config.start_drawdown, 1e-12)
            scale = 1.0 - min(1.0, max(0.0, (dd - config.start_drawdown) / denom))
            dynamic_cap = max(config.min_cap, base_pos[i] * scale)
            desired = min(base_pos[i], dynamic_cap)
            cap = dynamic_cap
            active_days += 1
        else:
            desired = base_pos[i]
            cap = 1.0
        if abs(desired - current) < config.trade_band:
            desired = current
        if desired < base_pos[i] - 1e-9:
            changed_days += 1
        current = float(np.clip(desired, 0.0, 1.0))
        positions[i] = current
        guard_cap[i] = cap
        guard_active[i] = dd >= config.start_drawdown
    label = f"resid_start{config.start_drawdown:.3f}_full{config.full_drawdown:.3f}_cap{config.min_cap:.3f}_band{config.trade_band:.2f}_rel{config.release_drawdown:.2f}"
    out = nav_from_positions(source, positions, label)
    out["residual_guard_base_position"] = base_pos
    out["residual_guard_account_drawdown"] = account_dd
    out["residual_guard_dynamic_cap"] = guard_cap
    out["residual_guard_active"] = guard_active
    out["residual_guard_changed"] = out["position"] < out["residual_guard_base_position"] - 1e-9
    meta = {
        "label": label,
        "start_drawdown": config.start_drawdown,
        "full_drawdown": config.full_drawdown,
        "min_cap": config.min_cap,
        "trade_band": config.trade_band,
        "release_drawdown": config.release_drawdown,
        "active_days": int(active_days),
        "changed_days": int(changed_days),
    }
    return out, meta


In [3]:
# Load and score current full-coverage path pool.
frames: dict[tuple[str, str], pd.DataFrame] = {}
base_rows = []
for family, paths in FAMILIES.items():
    for window, path in paths.items():
        df = load_daily(path)
        recalc = nav_from_positions(df, df["position"].to_numpy(), f"{family}_{window}_recalc")
        frames[(family, window)] = recalc
        row = {"family": family, "window": window, **metrics(recalc, f"{family}_{window}")}
        base_rows.append(row)

base_metrics = pd.DataFrame(base_rows).sort_values(["window", "max_dd", "relative_return"], ascending=[True, True, False])
base_metrics.to_csv(CARD / "current_base_metrics.csv", index=False)

display(Markdown("## Current Base Metrics"))
display(base_metrics)


## Current Base Metrics

,family,window,label,rows,start,end,return,buy_and_hold_return,relative_return,sharpe,max_dd,avg_pos,trades
1,recovery_reentry_2021_trendbreak_cap_v1,strict_oos,recovery_reentry_2021_trendbreak_cap_v1_strict...,1605,2020-01-02,2026-05-21,1023.710464,816.512500,207.197964,1.080314,30.436657,0.408919,207
9,recovery_reentry_conditional_vol_target_v1,strict_oos,recovery_reentry_conditional_vol_target_v1_str...,1605,2020-01-02,2026-05-21,2988.234589,816.512500,2171.722089,1.389946,30.923275,0.430109,212
29,soxl_soxx_efficiency_chop_guard_v1,strict_oos,soxl_soxx_efficiency_chop_guard_v1_strict_oos,1605,2020-01-02,2026-05-21,1411.719543,816.512500,595.207043,1.171697,31.957679,0.413345,210
13,recovery_reentry_cppi_floor_v1,strict_oos,recovery_reentry_cppi_floor_v1_strict_oos,1605,2020-01-02,2026-05-21,2410.985976,816.512500,1594.473476,1.285824,32.120180,0.440279,208
25,recovery_reentry_regime_drawdown_guard_v1,strict_oos,recovery_reentry_regime_drawdown_guard_v1_stri...,1605,2020-01-02,2026-05-21,2272.853019,816.512500,1456.340520,1.248022,32.120180,0.459580,212
17,recovery_reentry_equity_drawdown_guard_v1,strict_oos,recovery_reentry_equity_drawdown_guard_v1_stri...,1605,2020-01-02,2026-05-21,909.845863,816.512500,93.333364,1.075044,34.365251,0.390410,334
21,recovery_reentry_floor_v1,strict_oos,recovery_reentry_floor_v1_strict_oos,1605,2020-01-02,2026-05-21,2091.896203,816.512500,1275.383703,1.194099,41.587207,0.467687,202
5,recovery_reentry_anchor_confirm_v1,strict_oos,recovery_reentry_anchor_confirm_v1_strict_oos,1605,2020-01-02,2026-05-21,1894.611414,816.512500,1078.098915,1.164791,41.587207,0.466852,202
0,paper_model_oos_risk_check_v1,strict_oos,paper_model_oos_risk_check_v1_strict_oos,1605,2020-01-02,2026-05-21,1542.894521,816.512500,726.382021,1.108864,41.587207,0.462272,199
18,recovery_reentry_equity_drawdown_guard_v1,wf_2020,recovery_reentry_equity_drawdown_guard_v1_wf_2020,252,2020-01-02,2020-12-30,42.679646,58.093571,-15.413925,0.953160,24.373585,0.343285,40


In [4]:
SCREEN_FAMILIES = [
    "recovery_reentry_2021_trendbreak_cap_v1",
    "recovery_reentry_conditional_vol_target_v1",
    "recovery_reentry_cppi_floor_v1",
    "recovery_reentry_equity_drawdown_guard_v1",
    "soxl_soxx_efficiency_chop_guard_v1",
]
configs = []
for start_drawdown, full_drawdown, min_cap, trade_band, release_drawdown in product(
    [0.07, 0.09, 0.12, 0.15],
    [0.16, 0.20, 0.22, 0.24],
    [0.03, 0.05, 0.095, 0.15, 0.20],
    [0.00, 0.01, 0.02],
    [0.00, 0.03],
):
    if full_drawdown > start_drawdown + 0.02:
        configs.append(ResidualGuardConfig(float(start_drawdown), float(full_drawdown), float(min_cap), float(trade_band), float(release_drawdown)))

rows = []
selected_paths: dict[tuple[str, int, str], pd.DataFrame] = {}
for family in SCREEN_FAMILIES:
    paths = FAMILIES[family]
    has_wf = all(window in paths for window in ["wf_2020", "wf_2022", "wf_2023_2024"])
    for cfg_idx, cfg in enumerate(configs):
        strict_daily, strict_meta = apply_residual_guard(frames[(family, "strict_oos")], cfg)
        strict_metrics = metrics(strict_daily, f"{family}_{cfg_idx}_strict")
        wf_metrics = {}
        wf_meta = {}
        if has_wf:
            for wf_name in ["wf_2020", "wf_2022", "wf_2023_2024"]:
                wf_daily, meta = apply_residual_guard(frames[(family, wf_name)], cfg)
                wf_metrics[wf_name] = metrics(wf_daily, f"{family}_{cfg_idx}_{wf_name}")
                wf_meta[wf_name] = meta
        robust_score, strict_pass, full_pass = score_candidate(strict_metrics, wf_metrics)
        row = {
            "candidate_id": len(rows),
            "family": family,
            "has_walkforward": has_wf,
            **strict_meta,
            "strict_return": strict_metrics["return"],
            "strict_buy_and_hold_return": strict_metrics["buy_and_hold_return"],
            "strict_relative_return": strict_metrics["relative_return"],
            "strict_sharpe": strict_metrics["sharpe"],
            "strict_max_dd": strict_metrics["max_dd"],
            "strict_avg_pos": strict_metrics["avg_pos"],
            "strict_trades": strict_metrics["trades"],
            "robust_score": robust_score,
            "strict_pass": strict_pass,
            "full_pass": full_pass,
        }
        for wf_name in ["wf_2020", "wf_2022", "wf_2023_2024"]:
            m = wf_metrics.get(wf_name)
            meta = wf_meta.get(wf_name)
            if m:
                row.update({f"{wf_name}_{k}": v for k, v in m.items() if k not in {"label", "rows", "start", "end"}})
                row[f"{wf_name}_active_days"] = meta["active_days"]
                row[f"{wf_name}_changed_days"] = meta["changed_days"]
        rows.append(row)

screen = pd.DataFrame(rows).sort_values(["full_pass", "strict_pass", "robust_score"], ascending=[False, False, True]).reset_index(drop=True)
screen.to_csv(CARD / "residual_guard_refresh_screen.csv", index=False)
top = screen.head(100).copy()
top.to_csv(CARD / "residual_guard_refresh_top.csv", index=False)
family_summary = (
    screen.groupby("family")
    .agg(
        candidates=("candidate_id", "count"),
        strict_pass_count=("strict_pass", "sum"),
        full_pass_count=("full_pass", "sum"),
        best_robust_score=("robust_score", "min"),
        best_strict_relative=("strict_relative_return", "max"),
        best_strict_max_dd=("strict_max_dd", "min"),
    )
    .reset_index()
    .sort_values(["full_pass_count", "strict_pass_count", "best_robust_score"], ascending=[False, False, True])
)
family_summary.to_csv(CARD / "residual_guard_refresh_family_summary.csv", index=False)
print(f"configs={len(configs)} candidates={len(screen)} strict_pass={int(screen['strict_pass'].sum())} full_pass={int(screen['full_pass'].sum())}")
display(family_summary)
display(top.head(15))


configs=450 candidates=2250 strict_pass=0 full_pass=0


,family,candidates,strict_pass_count,full_pass_count,best_robust_score,best_strict_relative,best_strict_max_dd
1,recovery_reentry_conditional_vol_target_v1,450,0,0,25853.321175,-125.130239,18.859533
3,recovery_reentry_equity_drawdown_guard_v1,450,0,0,126868.335678,-322.504883,18.865896
2,recovery_reentry_cppi_floor_v1,450,0,0,139969.286982,-342.675874,17.789731
0,recovery_reentry_2021_trendbreak_cap_v1,450,0,0,181440.282064,-393.591224,19.335738
4,soxl_soxx_efficiency_chop_guard_v1,450,0,0,255899.661296,-474.050042,18.817867


,candidate_id,family,has_walkforward,label,start_drawdown,full_drawdown,min_cap,trade_band,release_drawdown,active_days,changed_days,strict_return,strict_buy_and_hold_return,strict_relative_return,strict_sharpe,strict_max_dd,strict_avg_pos,strict_trades,robust_score,strict_pass,full_pass,wf_2020_return,wf_2020_buy_and_hold_return,wf_2020_relative_return,wf_2020_sharpe,wf_2020_max_dd,wf_2020_avg_pos,wf_2020_trades,wf_2020_active_days,wf_2020_changed_days,wf_2022_return,wf_2022_buy_and_hold_return,wf_2022_relative_return,wf_2022_sharpe,wf_2022_max_dd,wf_2022_avg_pos,wf_2022_trades,wf_2022_active_days,wf_2022_changed_days,wf_2023_2024_return,wf_2023_2024_buy_and_hold_return,wf_2023_2024_relative_return,wf_2023_2024_sharpe,wf_2023_2024_max_dd,wf_2023_2024_avg_pos,wf_2023_2024_trades,wf_2023_2024_active_days,wf_2023_2024_changed_days
0,894,recovery_reentry_conditional_vol_target_v1,True,resid_start0.150_full0.240_cap0.200_band0.00_r...,0.15,0.24,0.2,0.00,0.00,795,656,691.382261,816.5125,-125.130239,1.052208,34.266345,0.316863,411,25853.321175,False,False,71.525200,58.093571,13.431629,1.255681,24.815273,0.368370,45,92,48,-20.520014,-86.588072,66.068058,-0.741938,22.204418,0.161872,24,76,14,67.254623,-0.356635,67.611258,1.678349,16.826953,0.258369,43,9,5
1,895,recovery_reentry_conditional_vol_target_v1,True,resid_start0.150_full0.240_cap0.200_band0.00_r...,0.15,0.24,0.2,0.00,0.03,795,656,691.382261,816.5125,-125.130239,1.052208,34.266345,0.316863,411,25853.321175,False,False,71.525200,58.093571,13.431629,1.255681,24.815273,0.368370,45,92,48,-20.520014,-86.588072,66.068058,-0.741938,22.204418,0.161872,24,76,14,67.254623,-0.356635,67.611258,1.678349,16.826953,0.258369,43,9,5
2,866,recovery_reentry_conditional_vol_target_v1,True,resid_start0.150_full0.220_cap0.200_band0.01_r...,0.15,0.22,0.2,0.01,0.00,801,641,671.171653,816.5125,-145.340847,1.041977,32.664660,0.316499,391,32482.539289,False,False,15.389034,58.093571,-42.704537,0.539836,22.705825,0.282158,94,161,117,-20.279512,-86.588072,66.308560,-0.731666,21.969013,0.161280,24,76,14,66.821551,-0.356635,67.178186,1.671423,16.826953,0.258106,40,9,10
3,867,recovery_reentry_conditional_vol_target_v1,True,resid_start0.150_full0.220_cap0.200_band0.01_r...,0.15,0.22,0.2,0.01,0.03,801,641,671.171653,816.5125,-145.340847,1.041977,32.664660,0.316499,391,32482.539289,False,False,15.389034,58.093571,-42.704537,0.539836,22.705825,0.282158,94,161,117,-20.279512,-86.588072,66.308560,-0.731666,21.969013,0.161280,24,76,14,66.821551,-0.356635,67.178186,1.671423,16.826953,0.258106,40,9,10
4,896,recovery_reentry_conditional_vol_target_v1,True,resid_start0.150_full0.240_cap0.200_band0.01_r...,0.15,0.24,0.2,0.01,0.00,796,656,670.450846,816.5125,-146.061654,1.042047,34.277285,0.315996,375,32787.322741,False,False,71.507884,58.093571,13.414313,1.255496,24.815273,0.368396,42,92,48,-20.520014,-86.588072,66.068058,-0.741938,22.204418,0.161872,24,76,14,67.206017,-0.356635,67.562651,1.677636,16.826953,0.258330,40,9,10
5,897,recovery_reentry_conditional_vol_target_v1,True,resid_start0.150_full0.240_cap0.200_band0.01_r...,0.15,0.24,0.2,0.01,0.03,796,656,670.450846,816.5125,-146.061654,1.042047,34.277285,0.315996,375,32787.322741,False,False,71.507884,58.093571,13.414313,1.255496,24.815273,0.368396,42,92,48,-20.520014,-86.588072,66.068058,-0.741938,22.204418,0.161872,24,76,14,67.206017,-0.356635,67.562651,1.677636,16.826953,0.258330,40,9,10
6,868,recovery_reentry_conditional_vol_target_v1,True,resid_start0.150_full0.220_cap0.200_band0.02_r...,0.15,0.22,0.2,0.02,0.00,800,643,670.224543,816.5125,-146.287956,1.041346,32.567919,0.316480,354,32801.877543,False,False,15.352013,58.093571,-42.741558,0.539167,22.714868,0.282597,89,161,117,-20.244838,-86.588072,66.343234,-0.729577,21.935074,0.161456,22,76,13,66.821551,-0.356635,67.178186,1.671423,16.826953,0.258106,40,9,10
7,869,recovery_reentry_conditional_vol_target_v1,True,resid_start0.150_full0.220_cap0.200_band0.02_r...,0.15,0.22,0.2,0.02,0.03,800,643,670.224543,816.5125,

In [5]:
selected = screen.iloc[0].to_dict()
selected_family = selected["family"]
selected_cfg = ResidualGuardConfig(
    float(selected["start_drawdown"]),
    float(selected["full_drawdown"]),
    float(selected["min_cap"]),
    float(selected["trade_band"]),
    float(selected["release_drawdown"]),
)
selected_metrics = []
selected_spans = []
selected_daily_paths = {}
for window in WINDOWS:
    if (selected_family, window) not in frames:
        continue
    daily, meta = apply_residual_guard(frames[(selected_family, window)], selected_cfg)
    m = metrics(daily, f"{selected_family}_{window}_selected")
    selected_metrics.append({"window": window, **m, **meta})
    selected_spans.append({"window": window, **max_drawdown_span(daily)})
    name = "selected_strict_oos_daily.csv" if window == "strict_oos" else f"selected_{window}_daily.csv"
    path = CARD / name
    daily.to_csv(path, index=False)
    selected_daily_paths[window] = str(path.relative_to(ROOT))

selected_metrics_df = pd.DataFrame(selected_metrics)
selected_spans_df = pd.DataFrame(selected_spans)
selected_metrics_df.to_csv(CARD / "selected_metrics.csv", index=False)
selected_spans_df.to_csv(CARD / "selected_drawdown_spans.csv", index=False)

strict_row = selected_metrics_df[selected_metrics_df["window"] == "strict_oos"].iloc[0]
strict_pass_count = int(screen["strict_pass"].sum())
full_pass_count = int(screen["full_pass"].sum())
if full_pass_count > 0:
    decision = "candidate_target_pass_needs_training_or_runtime_review"
    conclusion = [
        "At least one refreshed current-coverage residual guard clears strict and walk-forward gates.",
        "Do not promote directly; this should next be replayed through the canonical runtime/profile governance path and compared against paper trading constraints.",
    ]
elif strict_pass_count > 0:
    decision = "strict_candidate_walkforward_blocked"
    conclusion = [
        "A refreshed current-coverage residual guard can clear the strict full-window target, but none clears walk-forward guards.",
        "This is useful as a low-drawdown base candidate for diagnosis only; using it for model selection would still overfit the full window.",
    ]
else:
    decision = "diagnostic_no_current_coverage_low_dd_base"
    conclusion = [
        "No refreshed residual guard across the current full-coverage path pool clears the strict +30pp / <=15% target.",
        "This means the old 2026-03-31 low-drawdown base cannot be treated as current 2026-05 evidence without regenerating upstream low-risk actors or accepting stale coverage.",
    ]

blockers = []
if float(strict_row["relative_return"]) < STRICT_TARGET_REL:
    blockers.append(f"strict relative {strict_row['relative_return']:.2f}pp < {STRICT_TARGET_REL:.2f}pp")
if float(strict_row["max_dd"]) > STRICT_TARGET_DD:
    blockers.append(f"strict MaxDD {strict_row['max_dd']:.2f}% > {STRICT_TARGET_DD:.2f}%")
for wf_name, floor in WF_REL_FLOORS.items():
    rows = selected_metrics_df[selected_metrics_df["window"] == wf_name]
    if rows.empty:
        blockers.append(f"{wf_name} missing for selected family")
        continue
    row = rows.iloc[0]
    if row["relative_return"] < floor:
        blockers.append(f"{wf_name} relative {row['relative_return']:.2f}pp < {floor:.2f}pp")
    if row["max_dd"] > STRICT_TARGET_DD:
        blockers.append(f"{wf_name} MaxDD {row['max_dd']:.2f}% > {STRICT_TARGET_DD:.2f}%")
if blockers:
    conclusion.append("Primary blockers: " + "; ".join(blockers[:8]))
conclusion.append("Next valid step: if no full pass, refresh upstream actor-derived low-risk paths or move to training-time state-conditioned selection; do not keep tightening a path-level cap on the same high-drawdown families.")

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "role": "diagnose",
    "changed_layer": ["execution", "selection"],
    "candidate_count": int(len(screen)),
    "strict_pass_count": strict_pass_count,
    "full_pass_count": full_pass_count,
    "selected_candidate_id": int(selected["candidate_id"]),
    "selected_family": selected_family,
    "selected_config": {
        "start_drawdown": selected_cfg.start_drawdown,
        "full_drawdown": selected_cfg.full_drawdown,
        "min_cap": selected_cfg.min_cap,
        "trade_band": selected_cfg.trade_band,
        "release_drawdown": selected_cfg.release_drawdown,
    },
    "selected_metrics": selected_metrics_df.to_dict("records"),
    "selected_drawdown_spans": selected_spans_df.to_dict("records"),
    "external_sources": [
        "https://github.com/microsoft/qlib",
        "https://papers.ssrn.com/sol3/Delivery.cfm/SSRN_ID544742_code375170.pdf?abstractid=544742&mirid=1",
        "https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2308659",
        "https://arxiv.org/abs/1904.04912",
    ],
    "conclusion": conclusion,
    "artifacts": {
        "notebook": f"runtime/execution/card/{EXPERIMENT}/{EXPERIMENT}.ipynb",
        "current_base_metrics": f"runtime/execution/card/{EXPERIMENT}/current_base_metrics.csv",
        "screen": f"runtime/execution/card/{EXPERIMENT}/residual_guard_refresh_screen.csv",
        "top": f"runtime/execution/card/{EXPERIMENT}/residual_guard_refresh_top.csv",
        "family_summary": f"runtime/execution/card/{EXPERIMENT}/residual_guard_refresh_family_summary.csv",
        "selected_metrics": f"runtime/execution/card/{EXPERIMENT}/selected_metrics.csv",
        "selected_drawdown_spans": f"runtime/execution/card/{EXPERIMENT}/selected_drawdown_spans.csv",
        "selected_daily": selected_daily_paths,
        "selection": f"runtime/execution/card/{EXPERIMENT}/selection.json",
    },
}
(CARD / "selection.json").write_text(json.dumps(selection, indent=2, ensure_ascii=False) + chr(10), encoding="utf-8")

display(Markdown("## Selected Current-Coverage Residual Guard"))
display(selected_metrics_df)
display(Markdown("## Selected Drawdown Spans"))
display(selected_spans_df)
display(Markdown("## Conclusion"))
for item in conclusion:
    display(Markdown(f"- {item}"))
print(json.dumps({k: selection[k] for k in ["decision", "candidate_count", "strict_pass_count", "full_pass_count", "selected_family", "selected_config"]}, indent=2, ensure_ascii=False))


## Selected Current-Coverage Residual Guard

,window,label,rows,start,end,return,buy_and_hold_return,relative_return,sharpe,max_dd,avg_pos,trades,start_drawdown,full_drawdown,min_cap,trade_band,release_drawdown,active_days,changed_days
0,strict_oos,resid_start0.150_full0.240_cap0.200_band0.00_r...,1605,2020-01-02,2026-05-21,691.382261,816.512500,-125.130239,1.052208,34.266345,0.316863,411,0.15,0.24,0.2,0.0,0.0,795,656
1,wf_2020,resid_start0.150_full0.240_cap0.200_band0.00_r...,252,2020-01-02,2020-12-30,71.525200,58.093571,13.431629,1.255681,24.815273,0.368370,45,0.15,0.24,0.2,0.0,0.0,92,48
2,wf_2022,resid_start0.150_full0.240_cap0.200_band0.00_r...,250,2022-01-03,2022-12-29,-20.520014,-86.588072,66.068058,-0.741938,22.204418,0.161872,24,0.15,0.24,0.2,0.0,0.0,76,14
3,wf_2023_2024,resid_start0.150_full0.240_cap0.200_band0.00_r...,251,2024-01-02,2024-12-30,67.254623,-0.356635,67.611258,1.678349,16.826953,0.258369,43,0.15,0.24,0.2,0.0,0.0,9,5


## Selected Drawdown Spans

,window,peak_date,trough_date,max_dd,peak_position,trough_position
0,strict_oos,2021-01-21,2022-10-14,34.266345,0.797934,0.200000
1,wf_2020,2020-09-02,2020-09-08,24.815273,0.833066,0.200000
2,wf_2022,2022-03-29,2022-09-30,22.204418,0.187500,0.200000
3,wf_2023_2024,2024-03-07,2024-04-19,16.826953,0.350000,0.596478


## Conclusion

- No refreshed residual guard across the current full-coverage path pool clears the strict +30pp / <=15% target.

- This means the old 2026-03-31 low-drawdown base cannot be treated as current 2026-05 evidence without regenerating upstream low-risk actors or accepting stale coverage.

- Primary blockers: strict relative -125.13pp < 30.00pp; strict MaxDD 34.27% > 15.00%; wf_2020 MaxDD 24.82% > 15.00%; wf_2022 MaxDD 22.20% > 15.00%; wf_2023_2024 MaxDD 16.83% > 15.00%

- Next valid step: if no full pass, refresh upstream actor-derived low-risk paths or move to training-time state-conditioned selection; do not keep tightening a path-level cap on the same high-drawdown families.

{
  "decision": "diagnostic_no_current_coverage_low_dd_base",
  "candidate_count": 2250,
  "strict_pass_count": 0,
  "full_pass_count": 0,
  "selected_family": "recovery_reentry_conditional_vol_target_v1",
  "selected_config": {
    "start_drawdown": 0.15,
    "full_drawdown": 0.24,
    "min_cap": 0.2,
    "trade_band": 0.0,
    "release_drawdown": 0.0
  }
}
